In [1]:


import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

np.random.seed(42)
tf.random.set_seed(42)

In [3]:
df = pd.read_csv("C:\\Users\\bnajjar\\Desktop\\projet conda\\data\\hgw_long_term.csv", parse_dates=["timestamp"])

df = df.sort_values(["gateway_id", "timestamp"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Gateways:", df["gateway_id"].nunique())

Shape: (438000, 31)
Gateways: 5


In [4]:
FEATURES = [
    'cpu_load', 'mem_used_pct', 'ping_latency',
    'packet_loss', 'wan_status',
    'cpu_mean_24h', 'ram_mean_24h',
    'cpu_std_24h', 'ram_std_24h',
    'cpu_slope_6h', 'ram_slope_6h',
    'wan_instability_6h', 'health_score'
]

TARGET = "incident_in_72h"

df = df.dropna(subset=FEATURES + [TARGET])

In [7]:
n = len(df)

train_df = df.iloc[:int(n*0.7)]
val_df   = df.iloc[int(n*0.7):int(n*0.85)]
test_df  = df.iloc[int(n*0.85):]

print(len(train_df), len(val_df), len(test_df))

306558 65691 65691


In [8]:
scaler = StandardScaler()
scaler.fit(train_df[FEATURES])

def scale(df):
    return scaler.transform(df[FEATURES]).astype(np.float32)

In [9]:
LOOKBACK = 24   # 24h
STEP = 1

def build_sequences(df):

    X_scaled = scale(df)
    y = df[TARGET].values

    X_seq = []
    y_seq = []

    for i in range(LOOKBACK, len(df)):
        X_seq.append(X_scaled[i-LOOKBACK:i])
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)

X_tr, y_tr = build_sequences(train_df)
X_va, y_va = build_sequences(val_df)
X_te, y_te = build_sequences(test_df)

print(X_tr.shape, X_te.shape)

(306534, 24, 13) (65667, 24, 13)


In [13]:
def build_lstm():

    inp = Input(shape=(LOOKBACK, len(FEATURES)))

    x = LSTM(48, return_sequences=True, dropout=0.3)(inp)
    x = LSTM(24, return_sequences=False, dropout=0.3)(x)

    x = BatchNormalization()(x)

    x = Dense(24, activation='relu')(x)
    x = Dropout(0.4)(x)

    out = Dense(1, activation='sigmoid')(x)

    model = Model(inp, out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(curve="PR", name="prauc")
        ]
    )

    return model

In [14]:
es = EarlyStopping(
    monitor="val_prauc",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es, lr],
    verbose=1
)

Epoch 1/15
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 45s 19ms/step - auc: 0.9418 - loss: 0.1208 - prauc: 0.8893 - val_auc: 0.9289 - val_loss: 0.1641 - val_prauc: 0.8732 - learning_rate: 0.0010
Epoch 2/15
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 52s 21ms/step - auc: 0.9447 - loss: 0.1183 - prauc: 0.8930 - val_auc: 0.9326 - val_loss: 0.1657 - val_prauc: 0.8720 - learning_rate: 0.0010
Epoch 3/15
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 47s 20ms/step - auc: 0.9467 - loss: 0.1161 - prauc: 0.8964 - val_auc: 0.9311 - val_loss: 0.1537 - val_prauc: 0.8729 - learning_rate: 0.0010
Epoch 4/15
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 49s 20ms/step - auc: 0.9481 - loss: 0.1152 - prauc: 0.8977 - val_auc: 0.9229 - val_loss: 0.1592 - val_prauc: 0.8654 - learning_rate: 0.0010
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [16]:
from sklearn.metrics import precision_recall_curve, f1_score

y_prob = model.predict(X_te).flatten()

# courbe PR
precision, recall, thresholds = precision_recall_curve(y_te, y_prob)

f1_scores = 2 * precision * recall / (precision + recall + 1e-9)

best_idx = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.4f}")
print(f"Best F1: {f1_scores[best_idx]:.4f}")

# prédiction finale
y_pred = (y_prob > best_threshold).astype(int)

f1 = f1_score(y_te, y_pred)
print("Final F1:", f1)

2053/2053 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
Best threshold: 0.3101
Best F1: 0.8623
Final F1: 0.8622010260856998


In [17]:
model.save("C:\\Users\\bnajjar\\Desktop\\projet conda\\predictor\\long_horizon_dl\\lstm_3day.keras")

import joblib
joblib.dump(scaler, "C:\\Users\\bnajjar\\Desktop\\projet conda\\predictor\\long_horizon_dl\\lstm_scaler.pkl")

print("✅ modèle sauvegardé")

✅ modèle sauvegardé
